# Full manual lineage: Commercial electricity growth

Run each stage and inspect its output before continuing. The pinned 2026-09-26 publisher response and verified rebuild are supplied. Set `USE_PINNED_RAW = False` only for a deliberate fresh-source run; its values may differ. All processing and formulas are visible in this notebook. No portfolio module, launcher, or configuration file is required.


## 1 Source

Inspect the publisher URL and query below: https://www.eia.gov/electricity/data/eia861m/xls/sales_revenue.xlsx. The included raw response is a dated publisher payload with a SHA-256 digest, not the unsupported older portfolio snapshot. To acquire a new response, set `USE_PINNED_RAW = False` before running.

**Check before continuing:** publisher, selection, project folder, and pinned-response location must match this project; any new response must be preserved with its retrieval metadata.


In [1]:
import hashlib
import json
from datetime import UTC, date, datetime
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile
import numpy as np
import pandas as pd
import requests
from IPython.display import display

SLUG = "07_electricity_affordability"
FOLDER = SLUG.split("_", 1)[1]
SOURCE_URL = "https://www.eia.gov/electricity/data/eia861m/xls/sales_revenue.xlsx"
SNAPSHOT_DATE_OVERRIDE = None  # Leave None for the dated publisher response.
USE_PINNED_RAW = True
ROOT = next((candidate for candidate in (Path.cwd(), *Path.cwd().parents)
             if (candidate / "projects" / FOLDER / "README.md").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the extracted portfolio.")
PROJECT = ROOT / "projects" / FOLDER
RAW_DIR = PROJECT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
FETCH_LOG = []
PINNED_RESPONSES = {'52bbd7c793896191': 'f50ac9dcdcb1b8656e543927e9301bbb87288b1821c1d00ffa52e0902e77e4a0'}
print("Project:", PROJECT.relative_to(ROOT).as_posix())
print("Publisher:", SOURCE_URL)
print("Pinned response folder:", RAW_DIR.relative_to(ROOT).as_posix())

"""Monthly commercial energy surveillance; monthly MWh cannot identify peak MW."""


import numpy as np
import pandas as pd


def monthly_signals(frame: pd.DataFrame, *, growth_threshold: float = 5.0) -> pd.DataFrame:
    x = frame.copy()
    x["period"] = pd.to_datetime(x["period"], errors="raise")
    key = ["state", "sector", "period"]
    if x[key].isna().any().any() or x.duplicated(key).any():
        raise ValueError("State/sector/month grain must be complete and unique")
    if not x.period.dt.is_month_start.all():
        raise ValueError("Periods must be month starts")
    x = x.sort_values(key).reset_index(drop=True)
    for col in ["sales_mwh", "revenue_thousand"]:
        if (x[col].dropna() <= 0).any():
            raise ValueError(f"Positive {col} is required for price and growth decomposition")
    x["price_cents_kwh"] = 100 * x.revenue_thousand / x.sales_mwh
    previous = x[key + ["sales_mwh", "price_cents_kwh", "revenue_thousand"]].copy()
    previous["period"] += pd.DateOffset(years=1)
    previous = previous.rename(
        columns={c: f"prior_{c}" for c in ["sales_mwh", "price_cents_kwh", "revenue_thousand"]}
    )
    x = x.merge(previous, on=key, how="left", validate="one_to_one")
    x["sales_yoy_pct"] = 100 * (x.sales_mwh / x.prior_sales_mwh - 1)
    x["price_yoy_pct"] = 100 * (x.price_cents_kwh / x.prior_price_cents_kwh - 1)
    # Exact log decomposition: revenue growth = quantity effect + price effect.
    x["quantity_log_growth_pp"] = 100 * np.log(x.sales_mwh / x.prior_sales_mwh)
    x["price_log_growth_pp"] = 100 * np.log(x.price_cents_kwh / x.prior_price_cents_kwh)
    x["revenue_log_growth_pp"] = 100 * np.log(x.revenue_thousand / x.prior_revenue_thousand)
    x["calendar_month"] = x.period.dt.month
    seasonal = x.groupby(["state", "sector", "calendar_month"], observed=True)["sales_mwh"]
    x["historical_seasonal_mean"] = seasonal.transform(
        lambda s: s.shift().expanding(min_periods=3).mean()
    )
    x["historical_seasonal_sd"] = seasonal.transform(
        lambda s: s.shift().expanding(min_periods=3).std()
    )
    x["seasonal_z"] = (x.sales_mwh - x.historical_seasonal_mean) / x.historical_seasonal_sd.replace(
        0, np.nan
    )
    group = x.groupby(["state", "sector"], observed=True)
    x["rolling_growth_3m"] = group["sales_yoy_pct"].transform(
        lambda s: s.rolling(3, min_periods=3).mean()
    )
    # Guard row windows against gaps; this function does not create or fill a calendar.
    lag2 = group["period"].shift(2)
    consecutive = x.period.eq(lag2 + pd.DateOffset(months=2))
    x.loc[~consecutive, "rolling_growth_3m"] = np.nan
    x["above_threshold"] = x.sales_yoy_pct.ge(growth_threshold).astype(int)
    x["persistent_pressure_flag"] = (
        group["above_threshold"].transform(lambda s: s.rolling(3, min_periods=3).sum()).eq(3)
        & consecutive
    ).astype(int)
    return x


def persistence_sensitivity(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for threshold in [3.0, 5.0, 8.0]:
        signals = monthly_signals(frame, growth_threshold=threshold)
        latest = signals.loc[signals.period.eq(signals.period.max())]
        for row in latest.itertuples():
            rows.append(
                {
                    "state": row.state,
                    "period": row.period,
                    "growth_threshold_pct": threshold,
                    "persistent_pressure_flag": row.persistent_pressure_flag,
                    "sales_yoy_pct": row.sales_yoy_pct,
                    "seasonal_z": row.seasonal_z,
                }
            )
    return pd.DataFrame(rows)

def preserve_response(
    response: requests.Response,
    raw_root: Path,
    *,
    filename: str,
    publisher: str,
    **metadata: object,
) -> Path:
    response.raise_for_status()
    if Path(filename).name != filename:
        raise ValueError("Snapshot filename must be a basename")
    payload = response.content
    if not payload:
        raise ValueError("Cannot preserve an empty source snapshot")
    digest = hashlib.sha256(payload).hexdigest()
    directory = raw_root / digest
    directory.mkdir(parents=True, exist_ok=True)
    destination = directory / filename
    if destination.exists() and hashlib.sha256(destination.read_bytes()).hexdigest() != digest:
        raise ValueError("Content-addressed snapshot collision")
    if not destination.exists():
        destination.write_bytes(payload)
    record = {
        **metadata,
        "publisher": publisher,
        "direct_url": response.url,
        "sha256": digest,
        "bytes": len(payload),
        "accessed_at": datetime.now(UTC).isoformat(),
    }
    # Response headers are deliberately excluded from public provenance.
    metadata_path = directory / "metadata.json"
    if metadata_path.exists():
        saved = json.loads(metadata_path.read_text(encoding="utf-8"))
        if saved.get("sha256") != digest or saved.get("bytes") != len(payload):
            raise ValueError("Content-addressed snapshot metadata mismatch")
    else:
        metadata_path.write_text(json.dumps(record, indent=2), encoding="utf-8")
    return destination

def project_dir(slug):
    return ROOT / "projects" / slug.split("_", 1)[1]

def _get(url, **kwargs):
    prepared = requests.Request("GET", url, params=kwargs.get("params")).prepare()
    request_key = hashlib.sha256(prepared.url.encode("utf-8")).hexdigest()[:16]
    folder = RAW_DIR / "publisher_responses" / request_key
    if USE_PINNED_RAW:
        digest = PINNED_RESPONSES.get(request_key)
        if digest is None:
            raise FileNotFoundError("This publisher request has no pinned response")
        if len(digest) != 64 or any(ch not in "0123456789abcdef" for ch in digest):
            raise ValueError("Invalid pinned source checksum")
        body = (folder / f"{digest}.body").read_bytes()
        if hashlib.sha256(body).hexdigest() != digest:
            raise ValueError("Pinned publisher response checksum mismatch")
        metadata = json.loads((folder / f"{digest}.json").read_text(encoding="utf-8"))
        response = requests.Response()
        response.status_code = 200
        response.url = prepared.url
        response._content = body
    else:
        response = requests.get(url, timeout=90, **kwargs)
        response.raise_for_status()
        body = response.content
        if not body:
            raise ValueError("Publisher returned an empty response")
        digest = hashlib.sha256(body).hexdigest()
        folder.mkdir(parents=True, exist_ok=True)
        destination = folder / f"{digest}.body"
        if not destination.exists():
            destination.write_bytes(body)
        metadata = {"url": response.url, "sha256": digest, "bytes": len(body),
                    "accessed_at": datetime.now(UTC).isoformat()}
        (folder / f"{digest}.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    FETCH_LOG.append(metadata)
    return response

def parse_eia(workbook: bytes) -> pd.DataFrame:
    labels = pd.read_excel(BytesIO(workbook), sheet_name="Monthly-States", header=None, nrows=3)
    if str(labels.iloc[0, 8]).strip().upper() != "COMMERCIAL" or list(labels.iloc[1, 8:11]) != [
        "Revenue",
        "Sales",
        "Customers",
    ]:
        raise ValueError("EIA commercial-sector column contract changed")
    if list(labels.iloc[2, 8:11]) != ["Thousand Dollars", "Megawatthours", "Count"]:
        raise ValueError("EIA source units changed")
    raw = pd.read_excel(BytesIO(workbook), sheet_name="Monthly-States", header=2)
    if raw.shape[1] < 12:
        raise RuntimeError("EIA-861M workbook schema changed: expected at least 12 columns")
    commercial = raw.iloc[:, [0, 1, 2, 3, 8, 9, 10]].copy()
    commercial.columns = [
        "year",
        "month",
        "state",
        "data_status",
        "revenue_thousand",
        "sales_mwh",
        "customers",
    ]
    commercial["year"] = pd.to_numeric(commercial["year"], errors="coerce")
    commercial["month"] = pd.to_numeric(commercial["month"], errors="coerce")
    commercial = commercial.loc[
        commercial["state"].isin(["VA", "MD", "DC", "PA", "NJ"])
        & commercial["year"].ge(2022)
        & commercial["month"].between(1, 12)
    ].copy()
    commercial["period"] = pd.to_datetime(
        {
            "year": commercial["year"],
            "month": commercial["month"],
            "day": 1,
        },
        errors="coerce",
    )
    commercial["sector"] = "Commercial"
    return commercial[
        [
            "period",
            "state",
            "sector",
            "sales_mwh",
            "revenue_thousand",
            "customers",
            "data_status",
        ]
    ]

def _eia() -> pd.DataFrame:

    response = _get("https://www.eia.gov/electricity/data/eia861m/xls/sales_revenue.xlsx")
    frame = parse_eia(response.content)
    preserve_response(
        response,
        project_dir("07_electricity_affordability") / "data/raw",
        filename="sales_revenue.xlsx",
        publisher="US Energy Information Administration",
        source_url="https://www.eia.gov/electricity/data/eia861m/",
        coverage=f"{frame.period.min()} through {frame.period.max()}",
        grain="State, sector and month",
        refresh_frequency="Monthly, subject to revisions",
        reuse_basis="Public US government energy statistics",
        attribution="EIA Form 861M",
        limitations="Monthly sales are energy, not coincident peak demand. Preserve preliminary/revised data status; commercial sales do not isolate data centers.",
    )
    return frame


Project: projects/electricity_affordability
Publisher: https://www.eia.gov/electricity/data/eia861m/xls/sales_revenue.xlsx
Pinned response folder: projects/electricity_affordability/data/raw


## 2 Import

Run the import cell. It verifies the pinned response hash (or retrieves a new publisher response), parses it, and displays row count, columns, and a preview.

**Check before continuing:** no parse or source-integrity error; the imported table is nonempty, has the expected fields and grain, and the displayed source date matches the saved response metadata.


In [2]:
raw = _eia()
if raw.empty:
    raise ValueError("Publisher returned no source records")
SNAPSHOT_DATE = SNAPSHOT_DATE_OVERRIDE or (
    str(FETCH_LOG[0].get("accessed_at") or "")[:10] if FETCH_LOG else date.today().isoformat()
)
raw.to_csv(RAW_DIR / "source_snapshot.csv", index=False)
print("Snapshot date:", SNAPSHOT_DATE)
print("Publisher responses:", len(FETCH_LOG))
print("Imported rows:", len(raw), "columns:", len(raw.columns))
display(raw.head(3))


Snapshot date: 2026-09-26
Publisher responses: 1
Imported rows: 275 columns: 7


,period,state,sector,sales_mwh,revenue_thousand,customers,data_status
7,2026-07-01,DC,Commercial,684208.95,163363.05,27693.0,Preliminary
20,2026-07-01,MD,Commercial,2658310.70,465530.09,263015.0,Preliminary
31,2026-07-01,NJ,Commercial,3445433.10,660320.81,542433.0,Preliminary


## 3 Transform and clean

Run the original Commercial electricity growth transformation, embedded below. Inspect the fields, grain, and missing values before continuing.

**Check before continuing: inspect the cleaned preview, required fields, record grain, types, and missing values. Stop if a key or type rule fails.**


In [3]:
from dataclasses import asdict, dataclass

import pandas as pd


@dataclass(frozen=True)
class QualityReport:
    rows: int
    columns: int
    duplicate_rows: int
    missing_cells: int
    missing_rate: float
    passed: bool
    notes: list[str]


class DataQualityError(ValueError):
    """Raised when a dataset fails the required release checks."""


def profile(df: pd.DataFrame, key: str | None = None) -> dict:
    notes: list[str] = []
    duplicate_rows = int(df.duplicated().sum())
    missing_cells = int(df.isna().sum().sum())
    total_cells = max(df.shape[0] * df.shape[1], 1)
    if df.empty:
        notes.append("Dataset is empty")
    if key:
        if key not in df:
            notes.append(f"Required key '{key}' is missing")
        else:
            if df[key].isna().any():
                notes.append(f"Key '{key}' contains missing values")
            if df[key].duplicated().any():
                notes.append(f"Key '{key}' is not unique")
    if duplicate_rows:
        notes.append(f"Found {duplicate_rows} duplicate rows")
    report = QualityReport(
        rows=len(df),
        columns=len(df.columns),
        duplicate_rows=duplicate_rows,
        missing_cells=missing_cells,
        missing_rate=round(missing_cells / total_cells, 6),
        passed=not notes,
        notes=notes,
    )
    return asdict(report)


def require_passed(report: dict, project: str) -> None:
    if not report["passed"]:
        details = "; ".join(report["notes"]) or "unspecified quality failure"
        raise DataQualityError(f"Quality checks failed for {project}: {details}")


def require_columns(df: pd.DataFrame, columns: list[str]) -> None:
    missing = sorted(set(columns) - set(df.columns))
    if missing:
        raise ValueError(f"Required columns missing: {', '.join(missing)}")

def _snake(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = (
        out.columns.str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return out

def electricity(df: pd.DataFrame) -> pd.DataFrame:
    x = _snake(df)
    require_columns(x, ["period", "state", "sector", "sales_mwh", "revenue_thousand", "customers"])
    x["period"] = pd.to_datetime(x["period"], errors="coerce")
    for col in ["sales_mwh", "revenue_thousand", "customers"]:
        x[col] = pd.to_numeric(x[col], errors="coerce")
    x = monthly_signals(x)
    x["avg_price_cents_kwh"] = x["price_cents_kwh"]
    x["mwh_per_customer"] = x["sales_mwh"] / x["customers"].where(x["customers"] > 0)
    groups = x.groupby(["state", "sector"])
    prior_mean = groups["sales_mwh"].transform(
        lambda s: s.shift().rolling(12, min_periods=12).mean()
    )
    lag12 = groups["period"].shift(12)
    contiguous = x["period"].eq(lag12 + pd.DateOffset(months=12))
    x["rolling_peak_ratio"] = (x["sales_mwh"] / prior_mean).where(contiguous)
    x["data_center_market_flag"] = x["state"].isin(["VA", "MD", "DC", "PA", "NJ"]).astype(int)
    x["load_pressure_flag"] = x["persistent_pressure_flag"]
    x["load_pressure_index"] = 100 * (
        0.7 * x["rolling_growth_3m"].clip(-10, 20).add(10).div(30)
        + 0.3 * x["seasonal_z"].clip(-2, 3).add(2).div(5)
    )
    x["data_status"] = x.get("data_status", pd.Series(pd.NA, index=x.index))
    x["attribution_note"] = "Commercial monthly energy; not peak MW or data-center attribution"
    return x[
        [
            "period",
            "state",
            "sector",
            "sales_mwh",
            "revenue_thousand",
            "customers",
            "avg_price_cents_kwh",
            "mwh_per_customer",
            "sales_yoy_pct",
            "price_yoy_pct",
            "rolling_peak_ratio",
            "data_center_market_flag",
            "load_pressure_flag",
            "attribution_note",
            "load_pressure_index",
            "data_status",
            "rolling_growth_3m",
            "seasonal_z",
            "quantity_log_growth_pp",
            "price_log_growth_pp",
            "revenue_log_growth_pp",
        ]
    ]

clean = electricity(raw)
print("Clean rows:", len(clean), "columns:", len(clean.columns))
display(clean.head(3))


Clean rows: 275 columns: 21


,period,state,sector,sales_mwh,revenue_thousand,customers,avg_price_cents_kwh,mwh_per_customer,sales_yoy_pct,price_yoy_pct,...,data_center_market_flag,load_pressure_flag,attribution_note,load_pressure_index,data_status,rolling_growth_3m,seasonal_z,quantity_log_growth_pp,price_log_growth_pp,revenue_log_growth_pp
0,2022-01-01,DC,Commercial,630350.48,93870.99,26909.0,14.891873,23.425266,NaN,NaN,...,1,0,Commercial monthly energy; not peak MW or data...,NaN,Final,NaN,NaN,NaN,NaN,NaN
1,2022-02-01,DC,Commercial,558081.48,82440.97,26935.0,14.772210,20.719565,NaN,NaN,...,1,0,Commercial monthly energy; not peak MW or data...,NaN,Final,NaN,NaN,NaN,NaN,NaN
2,2022-03-01,DC,Commercial,584283.47,87888.79,27112.0,15.042149,21.550733,NaN,NaN,...,1,0,Commercial monthly energy; not peak MW or data...,NaN,Final,NaN,NaN,NaN,NaN,NaN


## 4 Calculate

Run the original analytical formulas, embedded below. Inspect the persistence_sensitivity output before continuing.

**Check before continuing: confirm the named result table is nonempty and its labels and units match the project description. Do not treat a fresh snapshot as the historical baseline.**


In [4]:
"""Decision-specific analytical routines used by notebooks and release checks."""


import re
from dataclasses import dataclass

import numpy as np
import pandas as pd

SEED = 20260904


@dataclass(frozen=True)
class AnalysisSpec:
    group: str
    metric: str
    score: str
    entity: str
    date: str | None
    higher_is_risk: bool = True


SPECS = {
    "01_consumer_complaints": AnalysisSpec(
        "product", "timely_flag", "response_risk_score", "complaint_id", "date_received", False
    ),
    "03_sec_financial_health": AnalysisSpec(
        "bank_name", "noncurrent_loan_pct", "composite_stress_score", "cert", "report_date"
    ),
    "04_hospital_readmissions": AnalysisSpec(
        "measure_name", "shrunk_readmission_ratio", "priority_score", "facility_id", None
    ),
    "05_drug_safety": AnalysisSpec(
        "therapeutic_category",
        "shortage_age_days",
        "escalation_score",
        "package_ndc",
        "update_date",
    ),
    "07_electricity_affordability": AnalysisSpec(
        "state", "sales_yoy_pct", "load_pressure_index", "state", "period"
    ),
    "08_power_plant_emissions": AnalysisSpec(
        "fuel", "co2_lb_mwh", "transition_priority_score", "plant_id", None
    ),
    "10_semiconductor_supply_chain": AnalysisSpec(
        "metric", "yoy_pct", "pressure_zscore", "series_id", "period"
    ),
}

def data_audit(frame: pd.DataFrame, entity: str) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "check": [
                "rows",
                "columns",
                "duplicate_entity_rows",
                "missing_cells",
                "numeric_columns",
            ],
            "value": [
                len(frame),
                frame.shape[1],
                int(frame.duplicated(entity).sum()) if entity in frame else 0,
                int(frame.isna().sum().sum()),
                len(frame.select_dtypes(include="number").columns),
            ],
        }
    )

def segment_scorecard(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    grouped = (
        frame.groupby(spec.group, dropna=False)
        .agg(
            records=(spec.entity, "size"),
            metric_mean=(spec.metric, "mean"),
            metric_median=(spec.metric, "median"),
            risk_score=(spec.score, "mean"),
        )
        .reset_index()
    )
    grouped["metric_gap_vs_total"] = grouped["metric_mean"] - frame[spec.metric].mean()
    grouped["risk_rank"] = (
        grouped["risk_score"].rank(method="dense", ascending=False).astype("Int64")
    )
    return grouped.sort_values(["risk_rank", "records"], ascending=[True, False])

def bootstrap_top_gap(frame: pd.DataFrame, spec: AnalysisSpec, draws: int = 1500) -> pd.DataFrame:
    rng = np.random.default_rng(SEED)
    scorecard = segment_scorecard(frame, spec)
    focus = scorecard.iloc[0][spec.group]
    a = frame.loc[frame[spec.group] == focus, spec.metric].dropna().to_numpy(float)
    b = frame.loc[frame[spec.group] != focus, spec.metric].dropna().to_numpy(float)
    if min(len(a), len(b)) < 2:
        return pd.DataFrame(
            {"focus_group": [focus], "gap": [np.nan], "ci_low": [np.nan], "ci_high": [np.nan]}
        )
    gaps = np.empty(draws)
    for i in range(draws):
        gaps[i] = (
            rng.choice(a, len(a), replace=True).mean() - rng.choice(b, len(b), replace=True).mean()
        )
    return pd.DataFrame(
        {
            "focus_group": [focus],
            "gap": [a.mean() - b.mean()],
            "ci_low": [np.quantile(gaps, 0.025)],
            "ci_high": [np.quantile(gaps, 0.975)],
            "bootstrap_draws": [draws],
        }
    )

def feature_diagnostics(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    numeric = frame.select_dtypes(include="number").drop(columns=[spec.score], errors="ignore")
    numeric = numeric.loc[:, numeric.nunique(dropna=True) > 1]
    if spec.score not in frame or numeric.empty:
        return pd.DataFrame(columns=["feature", "spearman_with_risk", "absolute_association"])
    corr = numeric.corrwith(frame[spec.score], method="spearman").dropna()
    return (
        corr.rename("spearman_with_risk")
        .rename_axis("feature")
        .reset_index()
        .assign(absolute_association=lambda d: d["spearman_with_risk"].abs())
        .sort_values("absolute_association", ascending=False)
        .head(12)
    )

def temporal_diagnostics(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    if not spec.date or spec.date not in frame:
        return pd.DataFrame(
            columns=["period", "records", "metric_mean", "risk_score", "rolling_risk"]
        )
    x = frame.dropna(subset=[spec.date]).copy()
    x["period"] = pd.to_datetime(x[spec.date]).dt.to_period("M").dt.to_timestamp()
    out = (
        x.groupby("period")
        .agg(
            records=(spec.entity, "size"),
            metric_mean=(spec.metric, "mean"),
            risk_score=(spec.score, "mean"),
        )
        .reset_index()
        .sort_values("period")
    )
    out["rolling_risk"] = out["risk_score"].rolling(3, min_periods=1).mean()
    out["risk_change"] = out["rolling_risk"].diff()
    return out

def sensitivity(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    rows = []
    for quantile in [0.70, 0.75, 0.80, 0.85, 0.90]:
        threshold = frame[spec.score].quantile(quantile)
        flagged = frame[frame[spec.score] >= threshold]
        rows.append(
            {
                "risk_quantile": quantile,
                "threshold": threshold,
                "flagged_records": len(flagged),
                "flagged_share": len(flagged) / len(frame),
                "flagged_metric_mean": flagged[spec.metric].mean(),
            }
        )
    return pd.DataFrame(rows)

def decision_queue(frame: pd.DataFrame, spec: AnalysisSpec, limit: int = 20) -> pd.DataFrame:
    columns = [spec.entity, spec.group, spec.metric, spec.score]
    if spec.date:
        columns.insert(1, spec.date)
    columns = list(dict.fromkeys(c for c in columns if c in frame))
    return frame.nlargest(limit, spec.score)[columns].reset_index(drop=True)

def energy_projection(frame: pd.DataFrame) -> pd.DataFrame:
    latest = frame.sort_values("period").groupby("state").tail(1).copy()
    rows = []
    for _, row in latest.iterrows():
        for annual_growth in [0.03, 0.06, 0.10]:
            rows.append(
                {
                    "state": row["state"],
                    "annual_growth": annual_growth,
                    "current_sales_mwh": row["sales_mwh"],
                    "projected_sales_mwh_3y": row["sales_mwh"] * (1 + annual_growth) ** 3,
                    "incremental_mwh_3y": row["sales_mwh"] * ((1 + annual_growth) ** 3 - 1),
                }
            )
    return pd.DataFrame(rows)

def run_analysis(slug: str, frame: pd.DataFrame) -> dict[str, pd.DataFrame]:
    spec = SPECS[slug]
    outputs = {
        "data_audit": data_audit(frame, spec.entity),
        "segment_scorecard": segment_scorecard(frame, spec),
        "bootstrap_gap": bootstrap_top_gap(frame, spec),
        "feature_diagnostics": feature_diagnostics(frame, spec),
        "temporal_diagnostics": temporal_diagnostics(frame, spec),
        "threshold_sensitivity": sensitivity(frame, spec),
        "decision_queue": decision_queue(frame, spec),
    }
    if slug == "01_consumer_complaints":
        outputs["response_queue_intervals"] = monthly_queue(frame)
    elif slug == "03_sec_financial_health":

        outputs["deposit_outflow_scenario"] = bank_scenario(frame)
        outputs["threshold_sensitivity"] = threshold_sensitivity(frame)
    elif slug == "04_hospital_readmissions":
        outputs["observed_rate_uncertainty"] = observed_rate_review(frame)
        outputs["shrinkage_sensitivity"] = shrinkage_sensitivity(frame)
    elif slug == "05_drug_safety":
        outputs["concentration_analysis"] = shortage_concentration(frame)
        outputs["record_hhi_sensitivity"] = concentration_sensitivity(frame)
    elif slug == "07_electricity_affordability":
        outputs["growth_scenarios"] = energy_projection(frame)
        outputs["monthly_pressure_signals"] = monthly_signals(frame)
        outputs["persistence_sensitivity"] = persistence_sensitivity(frame)
    elif slug == "08_power_plant_emissions":
        outputs["abatement_pareto"] = emissions_pareto(frame)
        outputs["peer_scenarios"] = peer_scenarios(
            frame.rename(columns={"generation_mwh": "net_generation_mwh"}).drop(
                columns=["peer_count"], errors="ignore"
            )
        )
    elif slug == "10_semiconductor_supply_chain":
        outputs["divergence_analysis"] = supply_chain_divergence(frame)
        outputs.update(pulse_tables(frame))
    return outputs

analysis_outputs = run_analysis(SLUG, clean)
print("Calculated outputs:", {name: len(frame) for name, frame in analysis_outputs.items()})
display(analysis_outputs["persistence_sensitivity"].head(5))


Calculated outputs: {'data_audit': 5, 'segment_scorecard': 5, 'bootstrap_gap': 1, 'feature_diagnostics': 12, 'temporal_diagnostics': 55, 'threshold_sensitivity': 5, 'decision_queue': 20, 'growth_scenarios': 15, 'monthly_pressure_signals': 275, 'persistence_sensitivity': 15}


,state,period,growth_threshold_pct,persistent_pressure_flag,sales_yoy_pct,seasonal_z
0,DC,2026-07-01,3.0,0,-6.895593,-1.844345
1,MD,2026-07-01,3.0,0,-6.427003,-1.145066
2,NJ,2026-07-01,3.0,0,-6.543688,-3.377849
3,PA,2026-07-01,3.0,0,1.728876,1.072385
4,VA,2026-07-01,3.0,1,13.554694,3.762481


## 5 Check

Run every assertion and inspect the quality report. These checks apply to the current response, not to unsupplied historical downloads.

**Check before continuing:** every assertion passes and `passed` is true. Investigate missing keys, duplicates, impossible values, and material row-count changes before release.


In [5]:
KEY_COLUMNS = ["state","sector","period"]
report = profile(clean)
assert report["passed"], report["notes"]
assert clean[KEY_COLUMNS].notna().all().all(), "Missing record key"
assert not clean.duplicated(KEY_COLUMNS).any(), "Duplicate record key"
assert len(raw) > 0 and len(clean) > 0
print("Current source rows:", len(raw), "clean rows:", len(clean))
print("Quality:", report)


Current source rows: 275 clean rows: 275
Quality: {'rows': 275, 'columns': 21, 'duplicate_rows': 0, 'missing_cells': 790, 'missing_rate': 0.136797, 'passed': True, 'notes': []}


## 6 Save

Run this cell only after the checks pass. It writes the dated extract, quality report, analytical tables, and source evidence under `data/manual_rebuild/`. If a fresh response has the same date as an existing rebuild, Save uses a source-digest suffix so the verified result is not overwritten; it refuses a conflicting partial folder.

**Check before continuing:** every named output exists in the dated folder; the source digest and reported row counts match the current response.


In [6]:
CURRENT_SOURCE_SHA = [download_report["sha256"]] if SLUG == "01_consumer_complaints" else [item["sha256"] for item in FETCH_LOG]
OUTPUT_DIR = PROJECT / "data" / "manual_rebuild" / SNAPSHOT_DATE
if OUTPUT_DIR.exists():
    prior_file = OUTPUT_DIR / "quality_report.json"
    prior_hashes = json.loads(prior_file.read_text(encoding="utf-8")).get("source_sha256") if prior_file.is_file() else None
    if isinstance(prior_hashes, str):
        prior_hashes = [prior_hashes]
    if prior_hashes != CURRENT_SOURCE_SHA:
        digest_suffix = hashlib.sha256(json.dumps(CURRENT_SOURCE_SHA).encode("utf-8")).hexdigest()[:12]
        OUTPUT_DIR = PROJECT / "data" / "manual_rebuild" / f"{SNAPSHOT_DATE}-{digest_suffix}"
        if OUTPUT_DIR.exists():
            raise FileExistsError(f"A different or partial rebuild already exists at {OUTPUT_DIR}; review it before saving")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
clean.to_csv(OUTPUT_DIR / "tableau_extract.csv", index=False, date_format="%Y-%m-%d %H:%M:%S")
report.update({"project": SLUG, "source": "EIA Form EIA-861M",
               "sample_mode": False, "snapshot_date": SNAPSHOT_DATE,
               "source_sha256": [item["sha256"] for item in FETCH_LOG]})
(OUTPUT_DIR / "quality_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
analysis_dir = OUTPUT_DIR / "analysis"
analysis_dir.mkdir(exist_ok=True)
for name, frame in analysis_outputs.items():
    frame.to_csv(analysis_dir / f"{name}.csv", index=False, date_format="%Y-%m-%d")
(analysis_dir / "manifest.json").write_text(json.dumps({
    "project": SLUG, "sample_mode": False,
    "artifacts": {name: len(frame) for name, frame in analysis_outputs.items()}
}, indent=2), encoding="utf-8")
(OUTPUT_DIR / "source_manifest.json").write_text(json.dumps({
    "source_url": SOURCE_URL, "snapshot_date": SNAPSHOT_DATE,
    "responses": FETCH_LOG
}, indent=2), encoding="utf-8")
print("Saved manual rebuild to:", OUTPUT_DIR.relative_to(ROOT).as_posix())


Saved manual rebuild to: projects/electricity_affordability/data/manual_rebuild/2026-09-26


### Manual MySQL handoff after Save

If you are demonstrating the CDC-style database step, open this project's `sql/README.md` now. In MySQL Workbench, run `create_table.sql`, stage this run's checked `tableau_extract.csv`, run a temporary path-edited copy of `load_extract.sql`, then run `check_load.sql` and `review_queries.sql`. The SQL files create and check a mirror of the saved extract; they do not replace the notebook's calculations.

**Visible output:** Workbench shows the loaded row count, distinct source keys, missing-key checks, and a small inspection query. **Check before Show:** loaded rows equal this run's `quality_report.json` rows, distinct keys equal loaded rows, missing keys equal zero, and no unexplained load warnings remain. For a fresh response, use its own dated folder, not the pinned counts. If MySQL is unavailable, continue to Show with the notebook evidence and say explicitly that the SQL mirror was not tested.


## 7 Show

Display the rebuilt analytical table below. The packaged Tableau workbook in this project's tableau folder displays the pinned 2026-09-26 extract and is an additional saved view. Compare the saved table and dashboard before citing either; a fresh publisher response requires a deliberate workbook refresh and caption review.

**Check before continuing:** the displayed rows come from the current `analysis_outputs` and saved extract, with labels and units checked against this project's README.


In [7]:
display(analysis_outputs["persistence_sensitivity"].head(10))
print("Tableau input:", (OUTPUT_DIR / "tableau_extract.csv").relative_to(ROOT).as_posix())


,state,period,growth_threshold_pct,persistent_pressure_flag,sales_yoy_pct,seasonal_z
0,DC,2026-07-01,3.0,0,-6.895593,-1.844345
1,MD,2026-07-01,3.0,0,-6.427003,-1.145066
2,NJ,2026-07-01,3.0,0,-6.543688,-3.377849
3,PA,2026-07-01,3.0,0,1.728876,1.072385
4,VA,2026-07-01,3.0,1,13.554694,3.762481
5,DC,2026-07-01,5.0,0,-6.895593,-1.844345
6,MD,2026-07-01,5.0,0,-6.427003,-1.145066
7,NJ,2026-07-01,5.0,0,-6.543688,-3.377849
8,PA,2026-07-01,5.0,0,1.728876,1.072385
9,VA,2026-07-01,5.0,0,13.554694,3.762481


Tableau input: projects/electricity_affordability/data/manual_rebuild/2026-09-26/tableau_extract.csv


## 8 Decide

The next cell records a source-dated decision, saved evidence, and an interpretation limit for the pinned response. Review and revise both statements if you acquire a fresh response; do not carry this conclusion forward automatically.

**Final check:** the current quality check passed, each evidence file exists, and the conclusion is no stronger than the analysis supports.


In [8]:
DECISION_TEXT = (
    "For the pinned 2026-09-26 EIA snapshot, the July 2026 default 5% "
    "three-consecutive-month screen flags no state. Virginia has 13.55% "
    "July sales growth and meets the 3% sensitivity scenario, so prioritize "
    "manual review of Virginia, not a capacity or affordability finding."
)
INTERPRETATION_LIMIT = (
    "Monthly commercial MWh and average revenue per kWh cannot establish "
    "hourly peak demand, data-center attribution, or household affordability. "
    "July 2026 observations are preliminary and may be revised."
)
EVIDENCE_FILES = ["quality_report.json", "analysis/persistence_sensitivity.csv"]
assert report["passed"], "Prior quality check must pass"
assert DECISION_TEXT.strip(), "Write a decision supported by the rebuilt evidence"
assert INTERPRETATION_LIMIT.strip(), "State an interpretation limit"
missing = [name for name in EVIDENCE_FILES if not (OUTPUT_DIR / name).is_file()]
assert not missing, f"Missing evidence files: {missing}"
print("Source snapshot:", SNAPSHOT_DATE)
print("Evidence:", EVIDENCE_FILES)
print("Decision:", DECISION_TEXT)
print("Limit:", INTERPRETATION_LIMIT)

Source snapshot: 2026-09-26
Evidence: ['quality_report.json', 'analysis/persistence_sensitivity.csv']
Decision: For the pinned 2026-09-26 EIA snapshot, the July 2026 default 5% three-consecutive-month screen flags no state. Virginia has 13.55% July sales growth and meets the 3% sensitivity scenario, so prioritize manual review of Virginia, not a capacity or affordability finding.
Limit: Monthly commercial MWh and average revenue per kWh cannot establish hourly peak demand, data-center attribution, or household affordability. July 2026 observations are preliminary and may be revised.
